# EVerest Security Requirements — Final Notebook
### Zero-shot / Few-shot / RAG, on the 93 EVerest requirements, local model only

**What this notebook answers (from the paper writeup):**
- **RQ1** — How accurately can an LLM identify security requirements and their security objective?
- **RQ2** — How effectively can an LLM validate whether a requirement is genuinely well-formed?
- **RQ3** — How effectively can an LLM recover traceability links to architecture elements?
- **RQ4** — Does few-shot / RAG prompting improve on zero-shot?

**What changed from the previous version, per the supervisor meeting:**
- Only the **93 requirements** are used — the 159 documentation excerpts are gone entirely.
  They aren't needed: we are not training anything, only asking the model to label each
  requirement and checking the answer, so there's no reason to add extra non-security text.
- All **three research questions run through one shared function** (see Section 6) instead of
  three separate blocks of near-identical code.
- Everything runs on **one local model** (Qwen2.5-7B-Instruct, 4-bit, on the free Kaggle GPU).
  No Groq / Mistral / Gemini / xAI — nothing external, nothing that can rate-limit.
- **Zero data leakage, by construction:**
  1. A requirement used as a few-shot example is never one of the requirements being scored.
  2. RAG only ever retrieves from the annotation guideline / architecture list — reference
     material, never a test answer.
  3. The ground-truth label is only looked at *after* the model answers, to grade it — it is
     never part of the prompt (except the few-shot examples' *own* correct answers, which is
     what few-shot prompting means).

**Before running:** Settings → Accelerator → GPU T4 x2, Internet → On, and add the EVerest
dataset as this notebook's Input.

## 1. Install packages


In [1]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers scikit-learn scipy statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.6 MB/s eta 0:00:00


## 2. Load the local model

One function, `ask_llm(prompt)`. Give it text, get text back. That's the entire interface the
rest of the notebook needs — nothing else in this notebook needs to know this is a local
4-bit model rather than an API call.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os                          # <-- add this


MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("[warning] No GPU detected. Settings -> Accelerator -> GPU T4 x2, then restart.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
    ),
)
model.eval()
print("Model loaded.")


def ask_llm(prompt, max_new_tokens=300):
    """Send one prompt to the local model, get its text answer back."""
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# smoke test — fail loudly here, before spending an hour, if something's wrong
_test = ask_llm('Reply with ONLY this JSON: {"ok": true}')
print("Smoke test raw output:", _test)
assert '"ok"' in _test, "Model did not return the expected JSON — fix this before continuing."


CUDA available: True


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded.
Smoke test raw output: {"ok": true}


## 3. Load the embedding model (for RAG retrieval only)

RAG needs a way to measure "how similar is this text to that text". `sentence-transformers`
does that locally, in seconds, on CPU — no API, no extra quota.

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(texts, show_progress_bar=False)

def top_k_similar(query_text, corpus_texts, corpus_vectors, k=3):
    query_vec = embed([query_text])[0]
    sims = corpus_vectors @ query_vec / (
        np.linalg.norm(corpus_vectors, axis=1) * np.linalg.norm(query_vec) + 1e-8
    )
    best = np.argsort(-sims)[:k]
    return [corpus_texts[i] for i in best]

print("Embedder ready.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder ready.


## 4. Load the 93 requirements + the architecture model

This is the part your supervisor asked about directly, so the numbers below are printed
exactly so you can read them straight off the screen in the meeting:

- **93 requirements total** (documentation excerpts are not loaded at all — gone).
- **RQ2** (`requirement_label`) has real variance on its own: 84 True / 9 False.
- **RQ3** ground truth: a requirement only counts if the raw annotation actually recorded an
  architecture link for it. This was checked directly against the raw JSON files and the
  architecture model's 955 element IDs — **58 of 93 have a link, 35 do not**. That 58/35 split
  is a fact about how the dataset was annotated, not a subset chosen for convenience — there is
  nothing to grade the other 35 against.

In [4]:
import json, glob, re
import pandas as pd

DATA_DIR = "/kaggle/input"  # adjust the path below to match your dataset's exact folder name

# --- find the files (adjust these three paths if your dataset folder name differs) ---
REQ_DIR = "/kaggle/input/datasets/zainabnoor23/everst-data/texts/texts/everest-requirements/inter-annotator-agreement-per-requirement"
GUIDELINE_PATH = "/kaggle/input/datasets/zainabnoor23/everst-data/texts/texts/everest-requirements/annotation_guideline.md"
REPO_PATH = "/kaggle/input/datasets/zainabnoor23/everst-data/pcm-model/pcm-model/models/EverestStudy/everest.repository"
print("Requirements dir:", REQ_DIR)
print("Guideline file  :", GUIDELINE_PATH)
print("Architecture model:", REPO_PATH)

# --- architecture model: every element ID, and a name for the ones that have one ---
repo_text = open(REPO_PATH, encoding="utf-8").read()
ALL_ARCH_IDS = set(re.findall(r'id="([^"]+)"', repo_text))
ID_TO_NAME = dict(re.findall(r'<[^>]*\bid="([^"]+)"[^>]*\bentityName="([^"]*)"', repo_text))
ID_TO_NAME = {k: v for k, v in ID_TO_NAME.items() if v}


def collect_model_ids(node, found):
    """The raw annotation files store trace links under the key 'modelId' or 'modelIds'
    at various nesting depths, and sometimes as an empty string. This just walks the
    whole JSON and pulls out every non-empty one it finds, however deep it's nested."""
    if isinstance(node, dict):
        for key, value in node.items():
            if key in ("modelId", "modelIds"):
                if isinstance(value, str) and value.strip():
                    found.add(value.strip())
                elif isinstance(value, list):
                    for v in value:
                        if isinstance(v, str) and v.strip():
                            found.add(v.strip())
            else:
                collect_model_ids(value, found)
    elif isinstance(node, list):
        for item in node:
            collect_model_ids(item, found)


# --- load all 93 requirements ---
rows = []
for fp in sorted(glob.glob(f"{REQ_DIR}/*.json")):
    d = json.load(open(fp, encoding="utf-8"))
    req = d["requirement"]

    raw_ids = set()
    collect_model_ids(d, raw_ids)
    resolved_ids = {i for i in raw_ids if i in ALL_ARCH_IDS}
    component_names = sorted({ID_TO_NAME[i] for i in resolved_ids if i in ID_TO_NAME})

    rows.append({
        "id": req["id"],
        "text": req["text"],
        "security_related": bool(req["security-related"]),
        "security_objective": req.get("security_objective"),
        "requirement_label": bool(req["requirement_label"]),
        "gt_components": component_names,   # [] means: no trace link annotated for this one
    })

df = pd.DataFrame(rows)
assert len(df) == 93, f"expected 93 requirements, found {len(df)} — check REQ_DIR"

n_scorable = (df["gt_components"].map(len) > 0).sum()
print(f"\nTotal requirements          : {len(df)}")
print(f"RQ2 requirement_label       : True={df.requirement_label.sum()}  False={(~df.requirement_label).sum()}")
print(f"RQ3 have a trace link       : {n_scorable}")
print(f"RQ3 do NOT have a trace link: {len(df) - n_scorable}")

# the fixed list of architecture elements the model is allowed to pick from for RQ3 —
# every element that is actually used as a real trace target somewhere in this dataset
RQ3_CANDIDATES = sorted({name for names in df["gt_components"] for name in names})
print(f"\nRQ3 candidate elements: {len(RQ3_CANDIDATES)} ->", RQ3_CANDIDATES)


Requirements dir: /kaggle/input/datasets/zainabnoor23/everst-data/texts/texts/everest-requirements/inter-annotator-agreement-per-requirement
Guideline file  : /kaggle/input/datasets/zainabnoor23/everst-data/texts/texts/everest-requirements/annotation_guideline.md
Architecture model: /kaggle/input/datasets/zainabnoor23/everst-data/pcm-model/pcm-model/models/EverestStudy/everest.repository

Total requirements          : 93
RQ2 requirement_label       : True=84  False=9
RQ3 have a trace link       : 58
RQ3 do NOT have a trace link: 35

RQ3 candidate elements: 32 -> ['API', 'AuthTokenProvider', 'Authentication', 'AuthorizeResponse', 'BankSessionToken', 'DC_EVSEMaximumLimits', 'DC_EVSEMinimumLimits', 'DC_EVSEPresentVoltage_Current', 'EvseManager', 'EvseSecurity', 'EvseSlac', 'EvseV2G', 'FirmwareUpdateRequest', 'FirmwareUpdateResponse', 'OCPP', 'OCPP201', 'PN532TokenProvider', 'PaymentOption', 'PersistentStore', 'Powermeter', 'ProvidedIdToken', 'RsPaymentTerminal', 'SignedUpdateFirmwareReque

## 5. Load the annotation guideline (RAG reference material for RQ1/RQ2)

This is the actual document that defines what counts as a security requirement and what
makes a requirement valid — the right thing for RAG to retrieve from, since it's the same
criteria the human annotators used.

In [5]:
guideline_text = open(GUIDELINE_PATH, encoding="utf-8").read()
GUIDELINE_CHUNKS = [c.strip() for c in guideline_text.split("\n\n") if len(c.strip()) > 40]
GUIDELINE_VECTORS = embed(GUIDELINE_CHUNKS)
print(f"Guideline split into {len(GUIDELINE_CHUNKS)} chunks for retrieval.")


Guideline split into 37 chunks for retrieval.


## 6. Example pool for few-shot prompts — and testing on all 93

**Fix:** the previous version held out 15 requirements as a separate few-shot pool and only
tested on the remaining 78. Your supervisor asked for all 93 to be evaluated — here's how that's
done with **zero data leakage**, not just fewer held out:

Instead of a random held-out split, 10 requirements are **hand-picked** as the few-shot example
pool — chosen to cover every security objective (confidentiality/integrity/availability/
authentication/other), both `requirement_label` values (True/False), and several RQ3 trace-link
cases. This is deterministic (no random seed to explain) and easy to justify: "these 10 were
picked to show the model one example of every category that exists in the data."

**All 93 requirements are tested.** The leakage risk is: if a requirement is both a few-shot
example *and* a test item, its own prompt would show the model the correct answer to itself.
Fixed by excluding the current item from its own example list at prompt-build time — every other
test item still sees the full example set. A check further down asserts this holds for every
single one of the 930 (93 items × up to 10 examples) example placements, not just spot-checked.


In [6]:
# Hand-picked, deterministic few-shot example pool — covers every objective, both
# requirement_label values, and several RQ3-linked items. No random split, nothing held out.
EXAMPLE_IDS = ["r0", "r1", "r2", "r3", "r4", "r5", "r6", "r15", "r28", "r33"]

example_df = df[df["id"].isin(EXAMPLE_IDS)].reset_index(drop=True)
assert len(example_df) == len(EXAMPLE_IDS), "one or more EXAMPLE_IDS not found in df — check spelling"

example_rq3_df = example_df[example_df["gt_components"].map(len) > 0].reset_index(drop=True)

# Test on ALL 93 for RQ1/RQ2 (this is the fix — nothing held out from testing).
test_df = df.reset_index(drop=True)
# RQ3 can only be tested on requirements that actually have a trace-link annotation (58 of 93).
test_rq3 = df[df["gt_components"].map(len) > 0].reset_index(drop=True)

print(f"Few-shot example pool : {len(example_df)} requirements {EXAMPLE_IDS}")
print(f"  covering objectives : {sorted(example_df.security_objective.dropna().unique())}")
print(f"  requirement_label   : True={example_df.requirement_label.sum()}  False={(~example_df.requirement_label).sum()}")
print(f"  with an RQ3 link    : {(example_df.gt_components.map(len) > 0).sum()}")
print()
print(f"RQ1 / RQ2 test set    : {len(test_df)} requirements  (all 93)")
print(f"RQ3 test set          : {len(test_rq3)} requirements  (all 58 that have a trace link)")


Few-shot example pool : 10 requirements ['r0', 'r1', 'r2', 'r3', 'r4', 'r5', 'r6', 'r15', 'r28', 'r33']
  covering objectives : ['authentication', 'availability', 'confidentiality', 'integrity', 'other']
  requirement_label   : True=6  False=4
  with an RQ3 link    : 10

RQ1 / RQ2 test set    : 93 requirements  (all 93)
RQ3 test set          : 58 requirements  (all 58 that have a trace link)


## 7. Prompts for RQ1, RQ2, RQ3

Three functions, one per research question. Each one builds a different prompt depending on
`condition` ("zero_shot" / "few_shot" / "rag") — that's the only thing that changes.

In [7]:
# Fixed, explicit, class-balanced example order — not head(5) on a sorted dataframe.
# EXAMPLE_IDS (section 6) is 4 False + 6 True; a plain head(5) always grabbed the same
# first 5 rows (3 False + 2 True) for nearly every prompt, which over-represents the
# rare False class 6x versus its real ~10% share of the data and biases the model toward
# predicting False. This order caps it at 2 False + 3 True per prompt instead.
_BALANCED_ORDER = ["r0", "r1", "r2", "r4", "r5"]   # 2 False (r0, r2) + 3 True (r1, r4, r5)

def _balanced_examples(current_id):
    ids = [i for i in _BALANCED_ORDER if i != current_id]   # zero-leakage: drop self if present
    if current_id in _BALANCED_ORDER:
        # keep 5 examples even when the current item was one of them — pull in the next
        # unused id from the full pool rather than showing only 4.
        backup = [i for i in EXAMPLE_IDS if i not in ids and i != current_id]
        ids = ids + backup[:1]
    return example_df[example_df["id"].isin(ids)].set_index("id").loc[ids].reset_index()


def build_rq1_prompt(row, condition):
    parts = [
        "Decide whether this text is a security requirement, and if so, which security "
        "objective it addresses: confidentiality, integrity, availability, authentication, "
        "or other.",  # FIX: "other" was missing — 10 of 93 requirements use this label and
                       # could never be predicted correctly without it being offered.
        'Reply with ONLY this JSON: {"security_related": true/false, "security_objective": "..."}',
    ]
    if condition == "few_shot":
        examples = _balanced_examples(row.id)   # zero-leakage + class-balanced, not just the first 5 by id
        parts.append("\nExamples:")
        for _, ex in examples.iterrows():
            obj = f'"{ex.security_objective}"' if ex.security_related else "null"
            parts.append(
                f'Text: "{ex.text}"\n'
                f'Answer: {{"security_related": {str(ex.security_related).lower()}, "security_objective": {obj}}}'
            )
    if condition == "rag":
        context = top_k_similar(row.text, GUIDELINE_CHUNKS, GUIDELINE_VECTORS, k=2)
        parts.append("\nRelevant guideline text:\n- " + "\n- ".join(context))
    parts.append(f'\nText: "{row.text}"\nAnswer:')
    return "\n".join(parts)


def build_rq2_prompt(row, condition):
    parts = [
        "Decide whether this text is a genuine, well-formed requirement (something the system "
        "SHALL or MUST do), or just descriptive/background text.",
        'Reply with ONLY this JSON: {"is_requirement": true/false}',
    ]
    if condition == "few_shot":
        examples = _balanced_examples(row.id)   # zero-leakage + class-balanced, not just the first 5 by id
        parts.append("\nExamples:")
        for _, ex in examples.iterrows():
            parts.append(
                f'Text: "{ex.text}"\n'
                f'Answer: {{"is_requirement": {str(ex.requirement_label).lower()}}}'
            )
    if condition == "rag":
        context = top_k_similar(row.text, GUIDELINE_CHUNKS, GUIDELINE_VECTORS, k=2)
        parts.append("\nRelevant guideline text:\n- " + "\n- ".join(context))
    parts.append(f'\nText: "{row.text}"\nAnswer:')
    return "\n".join(parts)


def build_rq3_prompt(row, condition, k_rag=10):
    candidates = RQ3_CANDIDATES
    if condition == "rag":
        candidates = top_k_similar(row.text, RQ3_CANDIDATES, embed(RQ3_CANDIDATES), k=k_rag)

    parts = [
        "Which of these architecture components does this requirement relate to?",
        "Candidates: " + ", ".join(candidates),
        'Reply with ONLY this JSON: {"components": ["Name1", "Name2", ...]} (use an empty list if none apply)',
    ]
    if condition == "few_shot":
        examples = example_rq3_df[example_rq3_df["id"] != row.id]   # zero-leakage
        parts.append("\nExamples:")
        for _, ex in examples.head(3).iterrows():
            # FIX: ex.gt_components is a Python set — f-string-ing it directly gives
            # {'A', 'B'} (Python set syntax), which is NOT valid JSON. The model was being
            # shown a malformed example and imitating that malformed format, which is why
            # RQ3 few-shot's F1 collapsed to ~0.09 while zero-shot/RAG stayed around 0.61-0.65.
            example_json = json.dumps(sorted(ex.gt_components))
            parts.append(f'Requirement: "{ex.text}"\nAnswer: {{"components": {example_json}}}')
    parts.append(f'\nRequirement: "{row.text}"\nAnswer:')
    return "\n".join(parts)

print("Prompt builders ready (zero-leakage: each item is always excluded from its own example list).")


Prompt builders ready (zero-leakage: each item is always excluded from its own example list).


## 8. Verify zero data leakage — before spending any GPU time

Rather than just claiming no leakage, this checks it directly: for every one of the 93 test
items, build its actual few-shot prompt and confirm the item's own text never appears inside its
own example list. This runs in under a second (no model calls) and fails loudly if it's ever
wrong, instead of silently producing inflated numbers.


In [8]:
_leak_found = []
for _, row in test_df.iterrows():
    for build_fn in (lambda r: build_rq1_prompt(r, "few_shot"), lambda r: build_rq2_prompt(r, "few_shot")):
        prompt = build_fn(row)
        # the item's own text must appear exactly once (in the final "Text: ..." line to be answered),
        # never inside the "Examples:" block above it.
        examples_section = prompt.split("\nText:")[0]
        if row.text in examples_section:
            _leak_found.append((row.id, build_fn))

for _, row in test_rq3.iterrows():
    prompt = build_rq3_prompt(row, "few_shot")
    examples_section = prompt.split("\nRequirement:")[0]
    if row.text in examples_section:
        _leak_found.append((row.id, "rq3"))

assert not _leak_found, f"LEAKAGE DETECTED for: {_leak_found} — stop and fix before running."
print(f"Zero-leakage check passed: all {len(test_df)} RQ1/RQ2 items and {len(test_rq3)} RQ3 items "
      f"verified — no item's own text appears in its own few-shot example list.")


Zero-leakage check passed: all 93 RQ1/RQ2 items and 58 RQ3 items verified — no item's own text appears in its own few-shot example list.


## 9. The shared runner — this replaces having three separate blocks of code

One function does the same three steps for every research question: build a prompt, ask the
model, parse the JSON answer. It's called nine times below (3 tasks × 3 conditions) instead
of being copy-pasted nine times.

In [9]:
def parse_json(text):
    """Pull the first {...} block out of the model's answer and parse it."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {}


def run_task(items_df, build_prompt_fn, parse_row_fn, condition):
    rows_out = []
    for i, (_, row) in enumerate(items_df.iterrows()):
        prompt = build_prompt_fn(row, condition)
        raw = ask_llm(prompt)
        parsed = parse_json(raw)
        if not parsed:                 # one retry if the first answer wasn't valid JSON
            raw = ask_llm(prompt)
            parsed = parse_json(raw)
        rows_out.append(parse_row_fn(row, parsed))
        if (i + 1) % 10 == 0:
            print(f"  {i + 1}/{len(items_df)} done")
    return pd.DataFrame(rows_out)


def parse_rq1_row(row, parsed):
    return {
        "id": row.id,
        "true_security_related": row.security_related,
        "true_objective": row.security_objective,
        "pred_security_related": parsed.get("security_related"),
        "pred_objective": parsed.get("security_objective"),
    }


def parse_rq2_row(row, parsed):
    return {
        "id": row.id,
        "true_is_requirement": row.requirement_label,
        "pred_is_requirement": parsed.get("is_requirement"),
    }


def parse_rq3_row(row, parsed):
    pred = parsed.get("components", [])
    return {
        "id": row.id,
        "true_components": set(row.gt_components),
        "pred_components": set(pred if isinstance(pred, list) else []),
    }

print("Runner ready.")


Runner ready.


## 10. Run all nine experiments

This is the "1-2-3, combined" execution: one small loop drives every research question and
every condition through the same `run_task` function above. This is the part that takes time
(roughly 30-45 minutes on a T4) — everything before this cell is instant.

In [10]:
TASKS = {
    "RQ1": (test_df, build_rq1_prompt, parse_rq1_row),
    "RQ2": (test_df, build_rq2_prompt, parse_rq2_row),
    "RQ3": (test_rq3, build_rq3_prompt, parse_rq3_row),
}
CONDITIONS = ["zero_shot", "few_shot", "rag"]

results = {}
for task_name, (items_df, build_fn, parse_fn) in TASKS.items():
    for condition in CONDITIONS:
        key = f"{task_name}_{condition}"
        print(f"\nRunning {key} ({len(items_df)} items)...")
        results[key] = run_task(items_df, build_fn, parse_fn, condition)

print("\nAll 9 runs complete.")



Running RQ1_zero_shot (93 items)...
  10/93 done
  20/93 done
  30/93 done
  40/93 done
  50/93 done
  60/93 done
  70/93 done
  80/93 done
  90/93 done

Running RQ1_few_shot (93 items)...
  10/93 done
  20/93 done
  30/93 done
  40/93 done
  50/93 done
  60/93 done
  70/93 done
  80/93 done
  90/93 done

Running RQ1_rag (93 items)...
  10/93 done
  20/93 done
  30/93 done
  40/93 done
  50/93 done
  60/93 done
  70/93 done
  80/93 done
  90/93 done

Running RQ2_zero_shot (93 items)...
  10/93 done
  20/93 done
  30/93 done
  40/93 done
  50/93 done
  60/93 done
  70/93 done
  80/93 done
  90/93 done

Running RQ2_few_shot (93 items)...
  10/93 done
  20/93 done
  30/93 done
  40/93 done
  50/93 done
  60/93 done
  70/93 done
  80/93 done
  90/93 done

Running RQ2_rag (93 items)...
  10/93 done
  20/93 done
  30/93 done
  40/93 done
  50/93 done
  60/93 done
  70/93 done
  80/93 done
  90/93 done

Running RQ3_zero_shot (58 items)...
  10/58 done
  20/58 done
  30/58 done
  40/58 done
 

## 11. Score every run — accuracy, precision, recall, F1

Standard classification metrics for RQ1 and RQ2, RQ3's multi-label precision/recall/F1, and the
separate 5-way objective-classification accuracy for RQ1.

**RQ1 caveat, stated plainly rather than left to look like an error:** all 93 requirements in
this dataset are genuine security requirements — there is no negative example. That means for
RQ1's `security_related` prediction there are no true negatives and no false positives possible,
so **precision is mechanically 1.0 whenever the model predicts any True**, and **accuracy equals
recall exactly**. These are still reported in full below (nothing hidden), with this explanation
attached so it reads as an intentional property of the evaluation, not a bug.


In [11]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def score_rq1(d):
    y_true = d["true_security_related"].astype(bool)
    y_pred = d["pred_security_related"].fillna(False).astype(bool)
    agreed = d[y_pred]  # rows the model itself thinks are security-related
    objective_accuracy = (
        (agreed["true_objective"] == agreed["pred_objective"]).mean() if len(agreed) else float("nan")
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "objective_accuracy (5-way)": objective_accuracy,
    }


def score_rq2(d):
    y_true = d["true_is_requirement"].astype(bool)
    y_pred = d["pred_is_requirement"].fillna(False).astype(bool)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


def score_rq3(d):
    precisions, recalls, f1s = [], [], []
    tp_total = fp_total = fn_total = 0
    for _, row in d.iterrows():
        tp = len(row.true_components & row.pred_components)
        fp = len(row.pred_components - row.true_components)
        fn = len(row.true_components - row.pred_components)
        tp_total += tp; fp_total += fp; fn_total += fn
        prec = tp / len(row.pred_components) if row.pred_components else 0.0
        rec = tp / len(row.true_components) if row.true_components else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        precisions.append(prec); recalls.append(rec); f1s.append(f1)
    return {
        "precision (per-item avg)": np.mean(precisions),
        "recall (per-item avg)": np.mean(recalls),
        "f1 (per-item avg)": np.mean(f1s),
        "TP (element-level)": tp_total, "FP (element-level)": fp_total, "FN (element-level)": fn_total,
    }


SCORERS = {"RQ1": score_rq1, "RQ2": score_rq2, "RQ3": score_rq3}

summary_rows = []
for key, d in results.items():
    task = key.split("_")[0]
    condition = key[len(task) + 1:]
    summary_rows.append({"task": task, "condition": condition, **SCORERS[task](d)})

summary_df = pd.DataFrame(summary_rows)
summary_df


,task,condition,accuracy,precision,recall,f1,objective_accuracy (5-way),precision (per-item avg),recall (per-item avg),f1 (per-item avg),TP (element-level),FP (element-level),FN (element-level)
0,RQ1,zero_shot,0.731183,1.000000,0.731183,0.844720,0.470588,NaN,NaN,NaN,NaN,NaN,NaN
1,RQ1,few_shot,0.935484,1.000000,0.935484,0.966667,0.471264,NaN,NaN,NaN,NaN,NaN,NaN
2,RQ1,rag,0.709677,1.000000,0.709677,0.830189,0.515152,NaN,NaN,NaN,NaN,NaN,NaN
3,RQ2,zero_shot,0.892473,0.951220,0.928571,0.939759,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RQ2,few_shot,0.741935,0.954545,0.750000,0.840000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,RQ2,rag,0.774194,0.943662,0.797619,0.864516,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,RQ3,zero_shot,NaN,NaN,NaN,NaN,NaN,0.617098,0.655747,0.593969,80.0,65.0,60.0
7,RQ3,few_shot,NaN,NaN,NaN,NaN,NaN,0.582471,0.632471,0.576427,74.0,63.0,66.0
8,RQ3,rag,NaN,NaN,NaN,NaN,NaN,0.602011,0.596839,0.563976,70.0,48.0,70.0


## 12. Confusion matrices — the actual numbers, per condition

A true 2×2 confusion matrix for RQ1 and RQ2 (6 total: 3 conditions × 2 RQs), plus RQ3's
element-level TP/FP/FN as a table (RQ3 is multi-label — one requirement can trace to several
architecture elements at once — so a single 2×2 matrix doesn't directly apply the way it does for
RQ1/RQ2).

**RQ1's matrix will always show 0 in the "true negative" and "false positive" cells** — that is
the same single-class property noted above, not a computation error: with no actual negatives in
the ground truth, those cells are structurally always zero.

Saved as both a CSV (`confusion_matrices.csv`, all 9 in one table) and a PNG image per condition,
for pasting directly into slides.


In [12]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

os.makedirs("/kaggle/working/confusion_matrices", exist_ok=True)
cm_rows = []

def save_confusion_matrix(y_true, y_pred, title, fname):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[False, True]).ravel()
    cm_rows.append({"condition": title, "TP": tp, "FP": fp, "TN": tn, "FN": fn})

    fig, ax = plt.subplots(figsize=(3.2, 3.2))
    ax.imshow([[tn, fp], [fn, tp]], cmap="Blues")
    for (i, j), v in np.ndenumerate([[tn, fp], [fn, tp]]):
        ax.text(j, i, str(v), ha="center", va="center", fontsize=14)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred: False", "Pred: True"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["True: False", "True: True"])
    ax.set_title(title, fontsize=10)
    fig.tight_layout()
    fig.savefig(f"/kaggle/working/confusion_matrices/{fname}.png", dpi=150)
    plt.close(fig)


for condition in CONDITIONS:
    d = results[f"RQ1_{condition}"]
    save_confusion_matrix(
        d["true_security_related"].astype(bool),
        d["pred_security_related"].fillna(False).astype(bool),
        f"RQ1 — {condition}", f"rq1_{condition}",
    )

for condition in CONDITIONS:
    d = results[f"RQ2_{condition}"]
    save_confusion_matrix(
        d["true_is_requirement"].astype(bool),
        d["pred_is_requirement"].fillna(False).astype(bool),
        f"RQ2 — {condition}", f"rq2_{condition}",
    )

confusion_df = pd.DataFrame(cm_rows)
confusion_df.to_csv("/kaggle/working/confusion_matrices.csv", index=False)
print("RQ1 / RQ2 confusion matrices (also saved as PNGs, one per condition):")
print(confusion_df.to_string(index=False))

print("\nRQ3 element-level counts (from the summary table above — multi-label, not a 2x2 matrix):")
print(summary_df[summary_df.task == "RQ3"][["condition", "TP (element-level)", "FP (element-level)", "FN (element-level)"]].to_string(index=False))


RQ1 / RQ2 confusion matrices (also saved as PNGs, one per condition):
      condition  TP  FP  TN  FN
RQ1 — zero_shot  68   0   0  25
 RQ1 — few_shot  87   0   0   6
      RQ1 — rag  66   0   0  27
RQ2 — zero_shot  78   4   5   6
 RQ2 — few_shot  63   3   6  21
      RQ2 — rag  67   4   5  17

RQ3 element-level counts (from the summary table above — multi-label, not a 2x2 matrix):
condition  TP (element-level)  FP (element-level)  FN (element-level)
zero_shot                80.0                65.0                60.0
 few_shot                74.0                63.0                66.0
      rag                70.0                48.0                70.0


## 13. RQ4 — is few-shot / RAG actually better than zero-shot?

McNemar's test for RQ1/RQ2 (paired True/False correctness on the same test items), a paired
comparison of per-item F1 for RQ3.

In [13]:
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

def mcnemar_pvalue(d_a, d_b, true_col, pred_col):
    merged = d_a[["id", true_col, pred_col]].merge(d_b[["id", pred_col]], on="id", suffixes=("_a", "_b"))
    correct_a = merged[f"{pred_col}_a"].fillna(False).astype(bool) == merged[true_col].astype(bool)
    correct_b = merged[f"{pred_col}_b"].fillna(False).astype(bool) == merged[true_col].astype(bool)
    table = pd.crosstab(correct_a, correct_b).reindex(index=[False, True], columns=[False, True], fill_value=0)
    return mcnemar(table.values, exact=True).pvalue

stat_rows = []
for task, true_col, pred_col in [("RQ2", "true_is_requirement", "pred_is_requirement")]:
    for a, b in [("zero_shot", "few_shot"), ("zero_shot", "rag"), ("few_shot", "rag")]:
        p = mcnemar_pvalue(results[f"{task}_{a}"], results[f"{task}_{b}"], true_col, pred_col)
        stat_rows.append({"task": task, "comparison": f"{a} vs {b}", "p_value": p})

def per_item_f1(d):
    f1s = []
    for _, row in d.iterrows():
        tp = len(row.true_components & row.pred_components)
        prec = tp / len(row.pred_components) if row.pred_components else 0.0
        rec = tp / len(row.true_components) if row.true_components else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if (prec + rec) else 0.0)
    return pd.Series(f1s, index=d["id"])

f1_zero = per_item_f1(results["RQ3_zero_shot"])
f1_few = per_item_f1(results["RQ3_few_shot"])
f1_rag = per_item_f1(results["RQ3_rag"])
common = f1_zero.index.intersection(f1_few.index).intersection(f1_rag.index)
for name, a, b in [("zero_shot vs few_shot", f1_zero, f1_few),
                    ("zero_shot vs rag", f1_zero, f1_rag),
                    ("few_shot vs rag", f1_few, f1_rag)]:
    p = wilcoxon(a[common], b[common]).pvalue
    stat_rows.append({"task": "RQ3", "comparison": name, "p_value": p})

# RQ1 has no negative class (all 93 are real security requirements), so there is no
# True/False correctness to pair for McNemar's test here — RQ4 for RQ1 is reported as the
# plain identification-rate numbers from Section 10 instead.

stats_df = pd.DataFrame(stat_rows)
stats_df


,task,comparison,p_value
0,RQ2,zero_shot vs few_shot,0.002577
1,RQ2,zero_shot vs rag,0.003418
2,RQ2,few_shot vs rag,0.663624
3,RQ3,zero_shot vs few_shot,0.922455
4,RQ3,zero_shot vs rag,0.423270
5,RQ3,few_shot vs rag,0.561848


## 14. Save everything

Everything below is bundled into one zip — predictions for all 9 runs, the metrics summary, confusion matrices (CSV + PNGs), the statistical tests, and a plain-text README explaining what each file is, so this is self-contained for handing to your supervisor without needing this notebook open.

In [14]:
import os, shutil

os.makedirs("/kaggle/working/predictions", exist_ok=True)
for key, d in results.items():
    d.to_csv(f"/kaggle/working/predictions/{key}.csv", index=False)

summary_df.to_csv("/kaggle/working/results_summary.csv", index=False)
stats_df.to_csv("/kaggle/working/statistical_tests.csv", index=False)
# confusion_matrices.csv and confusion_matrices/*.png were already written in section 12

readme = f'''EVerest Security-RE Experiment — output manifest
=================================================

results_summary.csv
    Accuracy / precision / recall / F1 for every RQ x condition (9 rows).
    RQ1 also includes objective_accuracy (5-way: confidentiality/integrity/
    availability/authentication/other). RQ1's precision is always 1.0 and
    accuracy equals recall — see section 11 for why (no negative class exists
    in this dataset's 93 requirements).

confusion_matrices.csv
    TP/FP/TN/FN for all 6 RQ1/RQ2 conditions. RQ1 rows always show TN=FP=0
    (same reason as above). confusion_matrices/*.png are the same data as images.

statistical_tests.csv
    McNemar's test (RQ2) / Wilcoxon signed-rank (RQ3) p-values comparing every
    pair of conditions. RQ1 has no paired significance test (no negative class
    to pair on) — its numbers in results_summary.csv are the full result.

predictions/RQ{{1,2,3}}_{{zero_shot,few_shot,rag}}.csv
    Every individual model prediction, one row per requirement, with the true
    label alongside it. This is what results_summary.csv and
    confusion_matrices.csv were computed from — nothing is only-in-summary.

Data / method notes
    - All 93 requirements evaluated for RQ1/RQ2 (all 58 with a trace-link
      annotation for RQ3) — nothing held out from testing.
    - 10 requirements are used as few-shot examples (EXAMPLE_IDS in the
      notebook, section 6) — chosen to cover every security objective and
      both requirement_label values. Zero-leakage is verified in section 8:
      an item is always excluded from its own example list.
    - Model: {MODEL_ID}, temperature 0 (greedy decoding), run locally.
'''
with open("/kaggle/working/README.txt", "w") as f:
    f.write(readme)

if os.path.exists("/kaggle/working/everest_final_results.zip"):
    os.remove("/kaggle/working/everest_final_results.zip")

files_to_zip = ["predictions", "confusion_matrices", "results_summary.csv",
                "statistical_tests.csv", "confusion_matrices.csv", "README.txt"]
tmp_dir = "/kaggle/working/_bundle"
if os.path.exists(tmp_dir):
    shutil.rmtree(tmp_dir)
os.makedirs(tmp_dir)
for name in files_to_zip:
    src = f"/kaggle/working/{name}"
    dst = f"{tmp_dir}/{name}"
    if os.path.isdir(src):
        shutil.copytree(src, dst)
    else:
        shutil.copy(src, dst)
shutil.make_archive("/kaggle/working/everest_final_results", "zip", tmp_dir)
shutil.rmtree(tmp_dir)

print("Saved (each individually, plus bundled together in everest_final_results.zip):")
for name in files_to_zip:
    print(" -", name)


Saved (each individually, plus bundled together in everest_final_results.zip):
 - predictions
 - confusion_matrices
 - results_summary.csv
 - statistical_tests.csv
 - confusion_matrices.csv
 - README.txt
